In [3]:
import pandas as pd
import numpy as np

In [6]:
df = pd.read_csv(r'D:\Work\Projects\Netflix\netflix_titles_cleaned.csv')

In [7]:
df.head()

,Show_Id,Category,Title,Director,Cast,Country,Release_Date,Rating,Duration,Type,Description,Date_Added_Clean,Year_Added,Month_Added
0,s1,TV Show,3%,Unknown,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,"August 14, 2020",TV-MA,4 Seasons,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...,2020-08-14,2020.0,August
1,s2,Movie,07:19,Jorge Michel Grau,"Demián Bichir, Héctor Bonilla, Oscar Serrano, ...",Mexico,"December 23, 2016",TV-MA,93 min,"Dramas, International Movies",After a devastating earthquake hits Mexico Cit...,2016-12-23,2016.0,December
2,s3,Movie,23:59,Gilbert Chan,"Tedd Chan, Stella Chung, Henley Hii, Lawrence ...",Singapore,"December 20, 2018",R,78 min,"Horror Movies, International Movies","When an army recruit is found dead, his fellow...",2018-12-20,2018.0,December
3,s4,Movie,9,Shane Acker,"Elijah Wood, John C. Reilly, Jennifer Connelly...",United States,"November 16, 2017",PG-13,80 min,"Action & Adventure, Independent Movies, Sci-Fi...","In a postapocalyptic world, rag-doll robots hi...",2017-11-16,2017.0,November
4,s5,Movie,21,Robert Luketic,"Jim Sturgess, Kevin Spacey, Kate Bosworth, Aar...",United States,"January 1, 2020",PG-13,123 min,Dramas,A brilliant group of students become card-coun...,2020-01-01,2020.0,January


In [8]:
df.isnull().sum()

Show_Id              0
Category             0
Title                0
Director             0
Cast                 0
Country              0
Release_Date         0
Rating               0
Duration             0
Type                 0
Description          0
Date_Added_Clean    10
Year_Added          10
Month_Added         10
dtype: int64

# 1. Handle Missing Values
# We fill missing categorical data with 'Unknown' or 'UR' (Unrated) to ensure they show up as a category in Power BI instead of 'Blank'.

In [9]:
df['Director'] = df['Director'].fillna('Unknown')
df['Cast'] = df['Cast'].fillna('Unknown')
df['Country'] = df['Country'].fillna('Unknown')
df['Rating'] = df['Rating'].fillna('UR')
df['Release_Date'] = df['Release_Date'].fillna('Unknown')

In [10]:
df.isnull().sum()

Show_Id              0
Category             0
Title                0
Director             0
Cast                 0
Country              0
Release_Date         0
Rating               0
Duration             0
Type                 0
Description          0
Date_Added_Clean    10
Year_Added          10
Month_Added         10
dtype: int64

# 2. Date Standardization
# The 'Release_Date' is currently a string. We convert it to a Datetime object.'errors=coerce' handles any weird date formats by setting them to NaT (Not a Time).

In [13]:
df['Date_Added_Clean'] = pd.to_datetime(df['Release_Date'].str.strip(), errors='coerce')

In [14]:
df['Date_Added_Clean']

0      2020-08-14
1      2016-12-23
2      2018-12-20
3      2017-11-16
4      2020-01-01
          ...    
7784   2020-10-19
7785   2019-03-02
7786   2020-09-25
7787   2020-10-31
7788   2020-03-01
Name: Date_Added_Clean, Length: 7789, dtype: datetime64[ns]

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7789 entries, 0 to 7788
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Show_Id           7789 non-null   object        
 1   Category          7789 non-null   object        
 2   Title             7789 non-null   object        
 3   Director          7789 non-null   object        
 4   Cast              7789 non-null   object        
 5   Country           7789 non-null   object        
 6   Release_Date      7789 non-null   object        
 7   Rating            7789 non-null   object        
 8   Duration          7789 non-null   object        
 9   Type              7789 non-null   object        
 10  Description       7789 non-null   object        
 11  Date_Added_Clean  7779 non-null   datetime64[ns]
 12  Year_Added        7779 non-null   float64       
 13  Month_Added       7779 non-null   object        
dtypes: datetime64[ns](1), fl

# 3. Feature Engineering (Year and Month)
# These columns are very helpful for creating 'Time Slicers'

In [16]:
df['Year_Added'] = df['Date_Added_Clean'].dt.year
df['Month_Added'] = df['Date_Added_Clean'].dt.month_name()

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7789 entries, 0 to 7788
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Show_Id           7789 non-null   object        
 1   Category          7789 non-null   object        
 2   Title             7789 non-null   object        
 3   Director          7789 non-null   object        
 4   Cast              7789 non-null   object        
 5   Country           7789 non-null   object        
 6   Release_Date      7789 non-null   object        
 7   Rating            7789 non-null   object        
 8   Duration          7789 non-null   object        
 9   Type              7789 non-null   object        
 10  Description       7789 non-null   object        
 11  Date_Added_Clean  7779 non-null   datetime64[ns]
 12  Year_Added        7779 non-null   float64       
 13  Month_Added       7779 non-null   object        
dtypes: datetime64[ns](1), fl

# 4. Duration Split
# The 'Duration' column contains text like '90 min' or '2 Seasons'. We need the number separately to perform calculations (like average movie length).

In [25]:
df['Duration_Value'] = df['Duration'].str.extract('(\d+)').astype(float)
df['Duration_Unit'] = df['Duration'].str.extract('([a-zA-Z]+)')

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7789 entries, 0 to 7788
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Show_Id           7789 non-null   object        
 1   Category          7789 non-null   object        
 2   Title             7789 non-null   object        
 3   Director          7789 non-null   object        
 4   Cast              7789 non-null   object        
 5   Country           7789 non-null   object        
 6   Release_Date      7789 non-null   object        
 7   Rating            7789 non-null   object        
 8   Duration          7789 non-null   object        
 9   Type              7789 non-null   object        
 10  Description       7789 non-null   object        
 11  Date_Added_Clean  7779 non-null   datetime64[ns]
 12  Year_Added        7779 non-null   float64       
 13  Month_Added       7779 non-null   object        
 14  Duration_Value    7789 n

# 5. Drop the old 'Release_Date' column since we have the standardized one.

In [18]:
df_cleaned = df.drop(columns=['Release_Date'])

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7789 entries, 0 to 7788
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Show_Id           7789 non-null   object        
 1   Category          7789 non-null   object        
 2   Title             7789 non-null   object        
 3   Director          7789 non-null   object        
 4   Cast              7789 non-null   object        
 5   Country           7789 non-null   object        
 6   Release_Date      7789 non-null   object        
 7   Rating            7789 non-null   object        
 8   Duration          7789 non-null   object        
 9   Type              7789 non-null   object        
 10  Description       7789 non-null   object        
 11  Date_Added_Clean  7779 non-null   datetime64[ns]
 12  Year_Added        7779 non-null   float64       
 13  Month_Added       7779 non-null   object        
dtypes: datetime64[ns](1), fl

# 6. Save the final cleaned version for Power BI

In [27]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

In [28]:
df['date_added_clean'] = pd.to_datetime(df['date_added_clean'], errors='coerce')

In [29]:
df.to_csv(r'D:\Work\Projects\Netflix\netflix_titles.csv', index=False)